# 🏥 Clinical Video Analysis — Treinamento YOLOv8
Treina 4 modelos especializados por contexto clínico:
- **Cirurgia**: detecção de sangramento
- **Consulta**: sinais de dor/desconforto
- **Fisioterapia**: análise de postura e movimento
- **Triagem**: linguagem corporal indicativa de violência

## 1. Instalar Dependências

In [54]:
import subprocess
subprocess.run(['pip', 'install', 'kagglehub', 'ultralytics', 'pyyaml'], check=True)
print('✅ Dependências instaladas')

✅ Dependências instaladas


## 2. Verificar GPU

In [55]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU disponível: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    DEVICE = 0
else:
    print('⚠️  GPU não encontrada — usando CPU (treinamento mais lento)')
    DEVICE = 'cpu'


✅ GPU disponível: NVIDIA GeForce RTX 3070
   VRAM: 8.6 GB


## 3. Baixar Datasets via KaggleHub

In [56]:
import kagglehub, os
from concurrent.futures import ThreadPoolExecutor, as_completed

DATASETS = {
    'cirurgia':     'jphajp/ur5e-srube-nurse-surgical-instruments',
    'consulta':     'coder98/emotionpain',
    'fisioterapia': 'sulaimanmuhammed/wlu-rehabilitation-posture',
    'triagem':      'simuletic/cctv-aggressive-poses-and-fight-detection-dataset',
}

def download(item):
    context, dataset_id = item
    try:
        path = kagglehub.dataset_download(dataset_id)
        return context, path, None
    except Exception as e:
        return context, None, str(e)

paths = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(download, item): item[0] for item in DATASETS.items()}
    for future in as_completed(futures):
        context, path, error = future.result()
        if error:
            print(f'❌ {context}: {error}')
        else:
            paths[context] = path
            print(f'✅ {context} → {path}')

print(f'\n{len(paths)}/4 datasets baixados')


✅ triagem → C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1
✅ cirurgia → C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1
✅ fisioterapia → C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
✅ consulta → C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1

4/4 datasets baixados


## 4. Inspecionar Estrutura dos Datasets

In [57]:
import os

for context, path in paths.items():
    print(f'\n📁 {context.upper()} — {path}')
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        if level > 2:
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        if level <= 1:
            subindent = '  ' * (level + 1)
            imgs = [f for f in files if f.endswith(('.jpg', '.png', '.jpeg'))]
            txts = [f for f in files if f.endswith('.txt')]
            if imgs:
                print(f'{subindent}🖼️  {len(imgs)} imagens')
            if txts:
                print(f'{subindent}📝 {len(txts)} labels')



📁 TRIAGEM — C:\Users\Xuanzang\.cache\kagglehub\datasets\simuletic\cctv-aggressive-poses-and-fight-detection-dataset\versions\1
1/
  Aggressive_Poses_Dataset/
    images/
    labels/

📁 CIRURGIA — C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1
1/
  UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8/
    📝 2 labels
    test/
    train/
    valid/
  UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8-obb/
    📝 2 labels
    test/
    train/
    valid/

📁 FISIOTERAPIA — C:\Users\Xuanzang\.cache\kagglehub\datasets\sulaimanmuhammed\wlu-rehabilitation-posture\versions\1
1/
  Blurred/
    Arm Raise Correct/
    Arm Raise Incorrect/
    Knee Extension Correct/
    Knee Extension Incorrect/
    Sit To Stand Correct/
    Sit To Stand Incorrect/
    train/
    val/

📁 CONSULTA — C:\Users\Xuanzang\.cache\kagglehub\datasets\coder98\emotionpain\versions\1
1/
  Frame_Labels/
    Frame_Labels/
  Images/
    Images/


## 5. Gerar data.yaml para Cada Contexto

In [ ]:
import os, shutil, yaml
from pathlib import Path
import cv2

os.makedirs('runs/processed', exist_ok=True)

def analyze_dataset(path):
    """Retorna dict com contagens reais"""
    p = Path(path)
    stats = {
        'videos': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.mp4','.mov','.avi'}),
        'images': sum(1 for x in p.rglob('*') if x.suffix.lower() in {'.jpg','.jpeg','.png'}),
        'labels': sum(1 for x in p.rglob('*.txt')),
        'subdirs': [d.name for d in p.iterdir() if d.is_dir()],
        'samples': [f.name for f in p.rglob('*')][:5]
    }
    return stats

# ANALISA TODOS
for context, path in paths.items():
    print(f"\n{context.upper()}:")
    stats = analyze_dataset(path)
    for k, v in stats.items():
        print(f"  {k:<10}: {v}")
    print()


🔍 ANÁLISE AUTOMÁTICA DOS DATASETS KAGGLE

TRIAGEM:
  videos    : 0
  images    : 103
  labels    : 103
  subdirs   : ['Aggressive_Poses_Dataset']
  samples   : ['Aggressive_Poses_Dataset', 'data.yaml', 'images', 'labels', 'labels.cache']


CIRURGIA:
  videos    : 0
  images    : 17960
  labels    : 17964
  subdirs   : ['UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8', 'UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8-obb']
  samples   : ['UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8', 'UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8-obb', 'data.yaml', 'README.dataset.txt', 'README.roboflow.txt']


FISIOTERAPIA:
  videos    : 339
  images    : 0
  labels    : 0
  subdirs   : ['Blurred']
  samples   : ['Blurred', 'Arm Raise Correct', 'Arm Raise Incorrect', 'Knee Extension Correct', 'Knee Extension Incorrect']


CONSULTA:
  videos    : 0
  images    : 48398
  labels    : 96796
  subdirs   : ['Frame_Labels', 'Images']
  samples   : ['Frame_Labels', 'Images', 'Frame_Labels', 'Images', 'FACS']



In [ ]:
# -----------------------------------------------------------
# PREPROCESSAMENTO COMPLETO DOS DATASETS PARA YOLOv8
# -----------------------------------------------------------

import os, cv2, yaml, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# ---------------------------
# Config
# ---------------------------
DATASET_OUT = Path("datasets")
DATASET_OUT.mkdir(exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png"}
VID_EXT = {".mp4", ".avi", ".mov"}

# ---------------------------
# Utilitários
# ---------------------------
def extract_frames(video_path, out_dir, step=5):
    cap = cv2.VideoCapture(str(video_path))
    frame_id = saved = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_id % step == 0:
            out = out_dir / f"{video_path.stem}_{frame_id:06d}.jpg"
            cv2.imwrite(str(out), frame)
            saved += 1
        frame_id += 1
    cap.release()
    return saved

def split_list(items, train_ratio=0.8):
    random.shuffle(items)
    n = int(len(items) * train_ratio)
    return items[:n], items[n:]

def detect_classes_from_labels(label_dir):
    """Detecta classes de labels de forma segura (ignora floats inválidos)"""
    classes = set()
    for lbl in Path(label_dir).rglob("*.txt"):
        with open(lbl) as f:
            for line in f:
                parts = line.split()
                if not parts: continue
                try:
                    c = int(float(parts[0]))  # converte float -> int
                    classes.add(c)
                except ValueError:
                    continue
    return sorted(classes)

# ---------------------------
# CIRURGIA
# ---------------------------
def process_cirurgia(path):
    print("🔧 Processando cirurgia")
    src = Path(path) / "UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8"
    out = DATASET_OUT / "cirurgia"
    shutil.copytree(src, out, dirs_exist_ok=True)
    yaml_path = out / "data.yaml"
    print("✅ cirurgia pronta")
    return "cirurgia", str(yaml_path)

# ---------------------------
# TRIAGEM
# ---------------------------
def process_triagem(path):
    print("🔧 Processando triagem")
    src = Path(path) / "Aggressive_Poses_Dataset"
    out = DATASET_OUT / "triagem"
    img_dir = src / "images"
    lbl_dir = src / "labels"

    imgs = list(img_dir.rglob("*.jpg"))
    train, val = split_list(imgs)

    for split, data in [("train", train), ("val", val)]:
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in data:
            lbl = lbl_dir / (img.stem + ".txt")
            if not lbl.exists(): continue
            shutil.copy2(img, out / "images" / split / img.name)
            shutil.copy2(lbl, out / "labels" / split / lbl.name)

    classes = detect_classes_from_labels(lbl_dir)
    if not classes: classes = [0]  # fallback para datasets pose/keypoints
    data_yaml = {
        "path": str(out),
        "train": "images/train",
        "val": "images/val",
        "nc": len(classes),
        "names": [f"class_{c}" for c in classes],
    }
    with open(out / "data.yaml", "w") as f:
        yaml.dump(data_yaml, f)

    print("✅ triagem pronta")
    return "triagem", str(out / "data.yaml")

# ---------------------------
# CONSULTA
# ---------------------------
def process_consulta(path):
    print("🔧 Processando consulta")
    out = DATASET_OUT / "consulta"
    img_dir = Path(path) / "Images" / "Images"
    lbl_dir = Path(path) / "Frame_Labels" / "Frame_Labels"

    imgs = list(img_dir.glob("*.jpg"))
    train, val = split_list(imgs)

    for split, data in [("train", train), ("val", val)]:
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)
        for img in data:
            lbl = lbl_dir / (img.stem + ".txt")
            if not lbl.exists(): continue
            shutil.copy2(img, out / "images" / split / img.name)
            shutil.copy2(lbl, out / "labels" / split / lbl.name)

    classes = detect_classes_from_labels(lbl_dir)
    if not classes: classes = [0]
    data_yaml = {
        "path": str(out),
        "train": "images/train",
        "val": "images/val",
        "nc": len(classes),
        "names": [f"class_{c}" for c in classes],
    }
    with open(out / "data.yaml", "w") as f:
        yaml.dump(data_yaml, f)

    print("✅ consulta pronta")
    return "consulta", str(out / "data.yaml")

# ---------------------------
# FISIOTERAPIA
# ---------------------------
def process_fisioterapia(path):
    print("🔧 Processando fisioterapia")
    out = DATASET_OUT / "fisioterapia"
    classes = [d for d in Path(path).iterdir() if d.is_dir()]

    for cls in classes:
        vids = list(cls.rglob("*.mp4"))
        temp = out / "temp" / cls.name
        temp.mkdir(parents=True, exist_ok=True)
        for vid in vids:
            extract_frames(vid, temp)
        frames = list(temp.glob("*.jpg"))
        train, val = split_list(frames)
        for split, data in [("train", train), ("val", val)]:
            target = out / split / cls.name
            target.mkdir(parents=True, exist_ok=True)
            for f in data:
                shutil.move(str(f), target / f.name)

    shutil.rmtree(out / "temp")
    print("✅ fisioterapia pronta")
    return "fisioterapia", str(out)

# ---------------------------
# EXECUÇÃO PARALLEL
# ---------------------------
def preprocess_all(paths):
    tasks = {
        "cirurgia": process_cirurgia,
        "triagem": process_triagem,
        "consulta": process_consulta,
        "fisioterapia": process_fisioterapia,
    }
    yamls = {}
    with ThreadPoolExecutor(max_workers=2) as ex:
        futures = {ex.submit(tasks[name], paths[name]): name for name in tasks}
        for f in futures:
            name, yaml_path = f.result()
            yamls[name] = yaml_path
            print(f"📄 {name} yaml → {yaml_path}")
    print("\n🎉 PREPROCESSAMENTO FINALIZADO")
    return yamls

# ---------------------------
# CHAMADA
# ---------------------------
# Usa os paths que você já definiu na célula anterior
yamls = preprocess_all(paths)
print("📄 YAMLs gerados:", yamls)

🔧 Processando cirurgia
🔧 Processando triagem
🔧 Processando consulta
🔧 Processando fisioterapia
✅ cirurgia pronta
📄 cirurgia yaml → datasets\cirurgia\data.yaml
✅ fisioterapia pronta


ValueError: invalid literal for int() with base 10: '0.000000'

## 6. Treinar os Modelos

In [62]:
from ultralytics import YOLO
import os
import traceback

# ================================
# CONFIG
# ================================

DEVICE = 0  # GPU (ou 'cpu')

BASE_DIR = "models/yolov8_custom"
os.makedirs(BASE_DIR, exist_ok=True)

resultados = {}
status = {}

# ================================
# SAFE TRAIN
# ================================

def safe_train(name, base_model, data_path, task, extra_params=None):
    """
    Treina modelo se não existir.
    Mantém toda a pasta de resultados do YOLO.
    """

    if extra_params is None:
        extra_params = {}

    exp_dir = f"{BASE_DIR}/{task}/{name}"

    # Skip se já treinado
    if os.path.exists(exp_dir):
        print(f"✅ {name.upper()} SKIPADO - já existe em {exp_dir}")
        resultados[name] = exp_dir
        return "SKIP"

    print("\n" + "="*60)
    print(f"🚀 INICIANDO TREINO: {name.upper()}")
    print("="*60)

    try:

        model = YOLO(base_model)

        common_params = {
            "data": data_path,
            "epochs": 50,
            "patience": 10,
            "augment": True,
            "workers": 2,
            "device": DEVICE,
            "exist_ok": True,
            "verbose": True,

            "task": task,

            "project": BASE_DIR,
            "name": name
        }

        # Config específica por task
        if task == "detect":
            common_params.update({
                "batch": 32,
                "imgsz": 640
            })

        elif task == "pose":
            common_params.update({
                "batch": 16,
                "imgsz": 640
            })

        elif task == "classify":
            common_params.update({
                "batch": 64,
                "imgsz": 224
            })

        # parâmetros extras
        common_params.update(extra_params)

        results = model.train(**common_params)

        save_dir = results.save_dir

        print(f"\n📁 Resultados salvos em:")
        print(save_dir)

        resultados[name] = save_dir

        return "TREINADO"

    except Exception as e:

        print(f"\n❌ ERRO no treino {name}")
        print(e)

        traceback.print_exc()

        return "ERRO"


# ================================
# PIPELINE
# ================================

print("\n🔄 PIPELINE COMPLETO YOLOv8\n")

status["cirurgia"] = safe_train(
    name="cirurgia",
    base_model="yolov8n.pt",
    data_path=yamls["cirurgia"],
    task="detect"
)

status["triagem_pose"] = safe_train(
    name="triagem_pose",
    base_model="yolov8n-pose.pt",
    data_path=yamls["triagem"],
    task="pose"
)

status["fisioterapia"] = safe_train(
    name="fisioterapia",
    base_model="yolov8n-cls.pt",
    data_path=classify_paths["fisioterapia"],
    task="classify"
)

status["consulta"] = safe_train(
    name="consulta",
    base_model="yolov8n-cls.pt",
    data_path=classify_paths["consulta"],
    task="classify"
)

# ================================
# RESUMO FINAL
# ================================

print("\n🎉 PIPELINE FINALIZADO")
print("="*60)

print("\n📊 STATUS DOS TREINOS:")
for k, v in status.items():
    print(f"{k:<15} → {v}")

print("\n📁 PASTAS GERADAS:")
for name, path in resultados.items():
    print(f"✅ {name:<15} → {path}")

print("\n🚀 Modelos prontos para inferência!")


🔄 PIPELINE COMPLETO YOLOv8


🚀 INICIANDO TREINO: CIRURGIA
Ultralytics 8.4.19  Python-3.12.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Xuanzang\.cache\kagglehub\datasets\jphajp\ur5e-srube-nurse-surgical-instruments\versions\1\UR5_Scrub_Nurse.v9-v7_migrate-dataset.yolov8/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode

KeyboardInterrupt: 

## 7. Avaliar Métricas dos Modelos

In [ ]:
from ultralytics import YOLO

print(f'{'Contexto':<15} {'mAP50':>8} {'mAP50-95':>10} {'Precisão':>10} {'Recall':>8}')
print('-' * 55)

for context, model_path in resultados.items():
    model = YOLO(model_path)
    metrics = model.val(data=yamls[context], verbose=False)
    print(
        f'{context:<15}'
        f'{metrics.box.map50:>8.3f}'
        f'{metrics.box.map:>10.3f}'
        f'{metrics.box.mp:>10.3f}'
        f'{metrics.box.mr:>8.3f}'
    )


## 8. Testar com Imagem Real

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

os.makedirs('data/samples', exist_ok=True)

for context, model_path in resultados.items():
    # Busca primeira imagem disponível do dataset
    test_img = next(
        (os.path.join(root, f)
         for root, _, files in os.walk(paths[context])
         for f in files if f.endswith(('.jpg', '.png', '.jpeg'))),
        None
    )

    if not test_img:
        print(f'⚠️  Nenhuma imagem encontrada para {context}')
        continue

    model = YOLO(model_path)
    results = model(test_img, conf=0.25)
    out_path = f'data/samples/resultado_{context}.jpg'
    results[0].save(out_path)

    print(f'\n🔍 {context.upper()}')
    print(f'   Imagem: {os.path.basename(test_img)}')
    print(f'   Detecções: {len(results[0].boxes)}')
    display(Image(filename=out_path, width=400))


## 9. Resumo Final

In [ ]:
import os

print('\n📦 MODELOS TREINADOS:')
print('-' * 45)
for context in ['cirurgia', 'consulta', 'fisioterapia', 'triagem']:
    path = f'models/yolov8_custom/{context}.pt'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ {context:<15} {path} ({size_mb:.1f} MB)')
    else:
        print(f'  ❌ {context:<15} não treinado')

print('\n🚀 Próximo passo: uvicorn app.main:app --reload')
